# Day 2 - Week 4: Hands-On Lab

### Step 1: Take a Week 3 model and evaluate it with 5-fold cross-validation using cross_val_score.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay


In [3]:
df = pd.read_csv("../../week1/day4/train.csv")

df["Age"] = df["Age"].fillna(df["Age"].mean())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

df = pd.get_dummies(
    df,
    columns=["Sex", "Embarked"],
    drop_first=True
)
target = "Survived"
droped_features = ['PassengerId','Cabin','Ticket','Name']
X = df.drop([target,*droped_features],axis=1)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


In [4]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=3))
])
scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=5,
    scoring="f1"
)
acc_scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy"
)
print("Accuracy scores:", acc_scores)
print("Mean Accuracy:", acc_scores.mean())
print("Standard Deviation:", acc_scores.std())

print("F1 scores:", scores)
print("Mean F1:", scores.mean())
print("F1 Standard Deviation:", scores.std())

Accuracy scores: [0.83216783 0.79020979 0.80985915 0.78873239 0.78873239]
Mean Accuracy: 0.8019403132079189
Standard Deviation: 0.017105054029457157
F1 scores: [0.75       0.71153846 0.73267327 0.7        0.71153846]
Mean F1: 0.7211500380807312
F1 Standard Deviation: 0.017872851104797808


### Step 2: Report the mean and standard deviation of the scores across folds.

In [7]:
print("F1 scores:", scores)
print("Mean F1 Score:", scores.mean())
print("Standard Deviation:", scores.std())

F1 scores: [0.83216783 0.7972028  0.80985915 0.78169014 0.84507042]
Mean F1 Score: 0.8131980695360979
Standard Deviation: 0.02295842084364427


### Step 3: Compare the cross-validated estimate to the single-split score from Day 1 and explain any difference.
first I wanna find the best k.

In [6]:
k_values = [3, 5, 7, 9, 11]
cv_results= []
for k in k_values:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="accuracy"
    )

    cv_results.append({
        "n_neighbors": k,
        "mean_accuracy": scores.mean(),
        "std_accuracy": scores.std()
    })
    cv_results_df = pd.DataFrame(cv_results)

cv_results_df

,n_neighbors,mean_accuracy,std_accuracy
0,3,0.801940,0.017105
1,5,0.803359,0.024849
2,7,0.806195,0.020048
3,9,0.799123,0.028937
4,11,0.813198,0.022958


In Day 1, I tested different values of n_neighbors using a single validation set.

The best value was:

- n_neighbors = 11
- Validation Accuracy = 0.8315

In Day 2, I tested the same values using 5-fold cross-validation. For each value of k, I calculated the mean validation accuracy across the five folds.

The best result was again:

- n_neighbors = 11
- Mean Cross-Validation Accuracy = 0.8132
- Standard Deviation = 0.0230

The cross-validation accuracy is slightly lower than the single validation score from Day 1.

This difference happens because the Day 1 score depends on only one validation split, while cross-validation evaluates the model on five different validation folds and averages the results.

Because of this, the cross-validation score gives a more stable and reliable estimate of how the model may perform on unseen data.

Both methods selected n_neighbors = 11 as the best value, which also gives more confidence in this choice.

### Step 4: Confirm Stratified Folds Are Used

For this classification task, I used **Stratified K-Fold** cross-validation.

Stratification keeps approximately the same class proportions in every fold.

For example, in my folds the target distribution was approximately:

* Class `1` → **62.6%**
* Class `0` → **37.4%**

These proportions stayed almost the same across all five folds.

This matters because the dataset is not perfectly balanced. If normal K-Fold created folds with very different class distributions, some folds could contain too many examples from one class and too few from the other.

That could make the validation scores less reliable.

Using stratified folds makes each validation fold more representative of the original dataset and gives a fairer evaluation of the classification model.


In [8]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), start=1):
    y_fold = y_train.iloc[val_idx]

    print(f"Fold {fold}")
    print(y_fold.value_counts(normalize=True))
    print()

Fold 1
Survived
0    0.622378
1    0.377622
Name: proportion, dtype: float64

Fold 2
Survived
0    0.622378
1    0.377622
Name: proportion, dtype: float64

Fold 3
Survived
0    0.626761
1    0.373239
Name: proportion, dtype: float64

Fold 4
Survived
0    0.626761
1    0.373239
Name: proportion, dtype: float64

Fold 5
Survived
0    0.619718
1    0.380282
Name: proportion, dtype: float64

